# Notebook 3: Retrieval experiments

**Goal:** compare three retrieval configurations on your real documents, see the scores at each stage, and gather the numbers that choose the "I can't find that" threshold.

| Config | What it does |
|---|---|
| A. vector only | embed the question, nearest chunks by cosine similarity |
| B. hybrid | vector search + keyword search, merged with Reciprocal Rank Fusion |
| C. hybrid + rerank | B, then a cross-encoder re-orders the candidates |

**Before you start:** the database is running, your documents are ingested (notebook 2), and your Voyage account has a payment method (the reranker needs it).

In [1]:
import sys
from pathlib import Path

# Make the project importable when the notebook runs from the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.ingestion import store
from app.retrieval import RetrievalConfig, retrieve

USER_ID = 1  # the development user created by schema.sql

CONFIGS = {
    "A vector only": RetrievalConfig(use_keyword=False, use_reranker=False),
    "B hybrid": RetrievalConfig(use_keyword=True, use_reranker=False),
    "C hybrid + rerank": RetrievalConfig(use_keyword=True, use_reranker=True),
}

for doc in store.list_documents(USER_ID):
    print(f"id {doc.id:>3} | {doc.filename} | {doc.chunk_count} chunks")

id   9 | Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx | 17 chunks


c:\Users\Dell\Downloads\rag-chatbot-step1\rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Your questions
These examples fit the guide document you ingested earlier. **Replace them with questions about your own corpus** as soon as you have chosen it. Include questions the documents cannot answer.

In [2]:
ANSWERABLE = [
    "Which vector database does the guide tell me to index the chunks in?",
    "How many passages should I retrieve before reranking?",
    "How do I verify that the dependencies are installed?",
]
UNANSWERABLE = [
    "What is the capital of Mars?",
    "How do I deploy the application to Kubernetes?",
]

## 2. One question, three configurations
Read the printed chunks. `sim` is the cosine similarity of the question and the chunk, `rerank` is the cross-encoder score (only config C), and `fused` is the Reciprocal Rank Fusion score.

In [3]:
def show(result) -> None:
    """Print the retrieved chunks with all their scores."""
    print(f"  {len(result.chunks)} chunks | candidates {result.candidates_considered} | "
          f"reranked {result.reranked} | relevant {result.relevant} | {result.timings_ms} ms")
    for rank, chunk in enumerate(result.chunks, start=1):
        rerank = f"{chunk.rerank_score:.3f}" if chunk.rerank_score is not None else "  -  "
        print(f"  {rank}. rerank {rerank} | sim {chunk.similarity:.3f} | fused {chunk.fused_score:.4f} | {chunk.heading}")
        print(f"     {chunk.text[:110]!r}")


question = ANSWERABLE[0]
print("QUESTION:", question)
for name, config in CONFIGS.items():
    print(f"\n[{name}]")
    show(await retrieve(question, USER_ID, config=config))

QUESTION: Which vector database does the guide tell me to index the chunks in?

[A vector only]
  6 chunks | candidates 17 | reranked False | relevant True | {'embed': 1226.1, 'search': 37.1} ms
  1. rerank   -   | sim 0.600 | fused 0.0164 | Phase 6 — Embeddings and indexing > Step 13: Index chunks in Qdrant
     'For each chunk, store its embedding and payload fields: user_id (for prototype use a single test user ID), doc'
  2. rerank   -   | sim 0.489 | fused 0.0161 | Recommended working order (do not skip ahead)
     'First complete Steps 1–6 to establish the project and UI. Then complete Steps 7–13 to ingest and store documen'
  3. rerank   -   | sim 0.488 | fused 0.0159 | How to work through this plan
     'Complete one step at a time. Do not move to the next step until the current step runs successfully.\n\nKeep a Gi'
  4. rerank   -   | sim 0.482 | fused 0.0156 | Portfolio Project 1
     'Practical Step-by-Step Build Guide: Document-Based RAG Chatbot\n\nGoal: Build a chatbot tha

## 3. Which heading ranks first, per configuration?
Where the three configurations disagree, check by eye which one picked the chunk that really answers the question.

In [14]:
for question in ANSWERABLE:
    print("\nQ:", question)
    for name, config in CONFIGS.items():
        result = await retrieve(question, USER_ID, config=config)
        top = result.chunks[0] if result.chunks else None
        print(f"  {name:<18} -> {top.heading if top else '(nothing found)'}")


Q: Which vector database does the guide tell me to index the chunks in?
  A vector only      -> Phase 6 — Embeddings and indexing > Step 13: Index chunks in Qdrant
  B hybrid           -> Phase 6 — Embeddings and indexing > Step 13: Index chunks in Qdrant
  C hybrid + rerank  -> Phase 6 — Embeddings and indexing > Step 13: Index chunks in Qdrant

Q: How many passages should I retrieve before reranking?
  A vector only      -> Phase 10 — Improve retrieval quality > Step 18: Add reranking
  B hybrid           -> Phase 10 — Improve retrieval quality > Step 18: Add reranking
  C hybrid + rerank  -> Phase 10 — Improve retrieval quality > Step 18: Add reranking

Q: How do I verify that the dependencies are installed?
  A vector only      -> Practical checklist
  B hybrid           -> Practical checklist
  C hybrid + rerank  -> Phase 1 — Set up the project > Step 5: Verify installation


## 4. Scores for answerable vs unanswerable questions
The service refuses to answer when even the best chunk scores too low. This table shows where answerable and unanswerable questions separate, so we can pick the thresholds (`min_similarity` and `min_rerank_score` in `app/config.py`).

In [15]:
for kind, questions in (("answerable", ANSWERABLE), ("unanswerable", UNANSWERABLE)):
    for question in questions:
        result = await retrieve(question, USER_ID, config=CONFIGS["C hybrid + rerank"])
        if not result.chunks:
            print(f"{kind:<13} no chunks | {question[:60]}")
            continue
        top_similarity = max(c.similarity for c in result.chunks)
        top_rerank = max((c.rerank_score or 0.0) for c in result.chunks)
        print(f"{kind:<13} sim {top_similarity:.3f} | rerank {top_rerank:.3f} | "
              f"relevant={result.relevant} | {question[:60]}")

answerable    sim 0.601 | rerank 0.745 | relevant=True | Which vector database does the guide tell me to index the ch
answerable    sim 0.599 | rerank 0.833 | relevant=True | How many passages should I retrieve before reranking?
answerable    sim 0.355 | rerank 0.806 | relevant=True | How do I verify that the dependencies are installed?
unanswerable  sim 0.146 | rerank 0.352 | relevant=True | What is the capital of Mars?
unanswerable  sim 0.520 | rerank 0.472 | relevant=True | How do I deploy the application to Kubernetes?


## Notes for DESIGN.md and EVAL.md (fill these in)
- Which configuration ranked the right chunk first most often?
- Did keyword search rescue any question that vector search missed (names, codes, numbers)?
- What top rerank score do answerable questions get, and what do unanswerable ones get? Is there a clean gap?
- How long does each stage take (embed, search, rerank)?